# 02h · Export the final V2 model (only if 02g says REPLACE)

Trains V2 once on **all** images (every fold's train + val + test) with the chosen recipe, for the median best
epoch found in cross-validation, and saves it in exactly the backend's format:
`state_dict` for `resnet50` + `fc = Sequential(Dropout(0.4), Linear(2048, 5))`, class order
`[Miner, Rust, Phoma, Healthy, Cerscospora]`, loadable with `torch.load(..., weights_only=True)`.

The file goes to `02_cnn/runs/export/` with its SHA-256 and the calibration temperature.
**Nothing in the PlantIQ backend is changed here** - swapping the file in is a separate, approved step.
Time: about 8-10 min.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "PLAN.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import hashlib
import os
import json

import numpy as np
import torch

from common import config, data, env
from common.train import TrainConfig, build_model, fit

config.ensure_dirs()
FORCE = os.getenv("PLANTIQ_FORCE_EXPORT") == "1"   # or set True to export even if the decision was "keep"
decision = json.loads((config.CNN_RESULTS / "decision.json").read_text())
if not (decision.get("replace") or FORCE):
    raise SystemExit("02g decided to KEEP the current model - nothing to export (set FORCE=True to export anyway).")
device = env.require_cuda()
env.seed_everything()

In [ ]:
cfg = TrainConfig(**json.loads((config.CNN_RESULTS / "v2_config.json").read_text()))
v2_summary = json.loads((config.CNN_RESULTS / "V2_summary.json").read_text())
bench = json.loads((config.CNN_RESULTS / "benchmark.json").read_text())
best_epochs = [f["best_epoch"] for f in v2_summary["folds"].values()]
cfg.epochs = int(np.median(best_epochs))
cfg.select, cfg.patience, cfg.name = "last", None, "V2_final"
temperature = float(np.median(bench["results"]["V2"]["temperatures"]))
print(f"training on all data for {cfg.epochs} epochs (CV best epochs {best_epochs}); temperature {temperature:.3f}")

uniq = data.load_unique()
images = data.load_images(uniq)
all_idx = np.arange(len(uniq)) if not config.SMOKE else data.fold_indices(uniq, data.load_splits(), 0)["train"]
tl = data.make_loader(images, uniq, all_idx, data.build_transform(cfg.train_aug, cfg.size), batch_size=cfg.batch_size,
                      train=True, sampler_mode=cfg.sampler, samples_per_epoch=cfg.samples_per_epoch, workers=cfg.workers)
# the 'validation' loader only prints progress here (these images are in the training data)
log_idx = all_idx[:: max(1, len(all_idx) // 500)]
vl = data.make_loader(images, uniq, log_idx, data.build_transform("eval", cfg.size), batch_size=128, train=False, workers=0)
res = fit(cfg, tl, vl, device, config.RUNS_DIR / "V2_final")

In [ ]:
out_dir = config.RUNS_DIR / "export"
out_dir.mkdir(parents=True, exist_ok=True)
out = out_dir / "best_resnet50_coffee_v2.pth"
state = {k: v.detach().float().cpu() for k, v in res["model"].state_dict().items()}
torch.save(state, out)

# Verify: load exactly like backend/app/services/cnn.py and predict a few images through its PIL pipeline
from PIL import Image
from torchvision import transforms

check = build_model("resnet50_dropout", pretrained=False)
check.load_state_dict(torch.load(str(out), map_location="cpu", weights_only=True))
check.eval()
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])])
sample = uniq.groupby("cls").head(4)
correct = 0
with torch.inference_mode():
    for p, lab in zip(sample["path"], sample["label"]):
        x = tf(Image.open(config.DATA_DIR / p).convert("RGB")).unsqueeze(0)
        correct += int(check(x).argmax(1).item() == lab)
print(f"backend-style reload OK; {correct}/{len(sample)} sample images correct (these are training images)")

sha = hashlib.sha256(out.read_bytes()).hexdigest()
info = {"file": out.name, "sha256": sha, "classes": config.CLASSES, "input": "224x224 RGB, ImageNet normalisation",
        "head": "Sequential(Dropout(0.4), Linear(2048, 5))", "temperature": temperature,
        "epochs": cfg.epochs, "recipe": cfg.to_dict(), "meta": env.run_meta()}
env.write_json(out_dir / "export_info.json", info)
(out_dir / f"{out.name}.sha256").write_text(f"{sha}  {out.name}\n")
print(json.dumps({k: info[k] for k in ("file", "sha256", "temperature", "epochs")}, indent=2))